# 🩻 NaijaCXR-VLM: Domain-Adapted Vision-Language Model for Chest X-Ray Report Generation

**NaijaCXR-VLM** is a domain-adapted multimodal architecture designed for accurate radiology report generation on low-resource, geographically diverse chest X-ray datasets (RSFUTH NaijaCXR) using **Google MedGemma-4B-IT** and **SigLIP-SO400M**.

---

### 🚀 Methodology Pipeline:
1. **Adversarial Domain Adaptation (DANN)**: Aligns the SigLIP vision encoder representation between high-resource source data (MIMIC-CXR) and target African clinical data (RSFUTH NaijaCXR) using a Gradient Reversal Layer (GRL).
2. **t-SNE & Domain Invariance Verification**: Validates visual domain shift reduction via t-SNE feature visualization and linear domain separability probes.
3. **Model Surgery & Multimodal Transplant**:
   - 4-bit NF4 Quantization of MedGemma-4B.
   - SigLIP Vision Tower transplantation.
   - Positional embedding interpolation ($64\times 64 \rightarrow 27\times 27$, 729 tokens for 384px images).
   - Projector surgery (pooling removal, 27x27 patch grid alignment).
4. **Supervised Fine-Tuning (SFT)**: Parameter-Efficient Fine-Tuning (LoRA) on standardized chest X-ray findings and impressions.
5. **Multi-Faceted Evaluation**: ROUGE-1/2/L, BLEU-1..4, BERTScore, and RadGraph F1 (clinical correctness).
6. **Explainability & Grounding**: Transformer Grad-CAM spatial localization for key clinical findings (*cardiomegaly, opacity, effusion*).
7. **Cross-Domain Evaluation on VQA-RAD**: Zero-shot/few-shot chest VQA benchmark.


## 1. Environment Setup & Dependencies


In [ ]:
# Install bleeding-edge transformers, PEFT, and quantization libraries
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U peft bitsandbytes accelerate datasets evaluate rouge_score bert_score radgraph
!pip install -q -U scikit-learn seaborn opencv-python matplotlib pillow pandas tqdm


## 2. Google Drive Mounting & Dataset Unpacking


In [ ]:
import os
import sys
from google.colab import drive, userdata

# Mount Google Drive
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        print("✅ Google Drive Mounted.")

# Optional: Hugging Face Authentication if needed
try:
    from huggingface_hub import login
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
        print("✅ Hugging Face Authenticated.")
except Exception as e:
    print(f"ℹ️ Hugging Face login note: {e}")

# Unpack Datasets (RSFUTH NaijaCXR & MIMIC-CXR)
!unzip -q -o /content/drive/MyDrive/RSFUTH_CXR.zip -d /content/
!unzip -q -o /content/drive/MyDrive/Seleccted_mimic_CXR_images.zip -d /content/
print("✅ Datasets extracted to /content/")


## 3. Dataset Standardization for Domain Adaptation


In [ ]:
#Standardize NaijaCXR Reports.CSV

import pandas as pd

naija = pd.read_csv("/content/New_CXR_Labels.csv")

naija_std = naija.rename(columns={
    "PID": "image_id",
    "Report": "findings",
    "Impression": "impression"
})

naija_std = naija_std[["image_id", "findings", "impression"]]
naija_std["domain"] = 1   # target

naija_std.to_csv("naija_standardized.csv", index=False)


#Standardize MimicCXR Reports.CSV

mimic = pd.read_csv("/content/Seleccted_mimic_CXR_images/My_Mimic_cxr_reports.csv")

mimic_std = mimic.rename(columns={
    "dicom_id": "image_id",
    "findings": "findings",
    "impression": "impression"
})

# Keep only what we need
mimic_std = mimic_std[["image_id", "findings", "impression"]]

mimic_std["domain"] = 0   # source domain

mimic_std.to_csv("mimic_standardized.csv", index=False)


## 4. Adversarial Domain Adaptation (SigLIP Vision Encoder)
We train a LoRA adapter on the SigLIP vision tower with an adversarial domain discriminator using a Gradient Reversal Layer (GRL).
This aligns the feature manifold between MIMIC-CXR and NaijaCXR.


In [ ]:
from transformers import AutoModel, AutoProcessor
from peft import LoraConfig, get_peft_model

# 1. Load the Official SigLIP Encoder from MedGemma
# MedGemma uses 'google/siglip-so400m-patch14-384' typically
SIGLIP_ID = "google/siglip-so400m-patch14-384"

processor = AutoProcessor.from_pretrained(SIGLIP_ID)
vision_tower = AutoModel.from_pretrained(SIGLIP_ID)

# 2. Freeze the Base Model
for param in vision_tower.parameters():
    param.requires_grad = False

# 3. Inject LoRA Adapters
# We target query and value projections in the Transformer
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

vision_tower = get_peft_model(vision_tower, peft_config)
vision_tower.print_trainable_parameters()
# Result: You will train ~0.5% of params (very memory efficient!)

vision_tower.to("cuda")
from transformers import SiglipVisionModel, AutoProcessor # Changed AutoModel to SiglipVisionModel

from peft import LoraConfig, get_peft_model

# 1. Load the Official SigLIP Encoder from MedGemma
# MedGemma uses 'google/siglip-so400m-patch14-384' typically
SIGLIP_ID = "google/siglip-so400m-patch14-384"

processor = AutoProcessor.from_pretrained(SIGLIP_ID)
vision_tower = AutoModel.from_pretrained(SIGLIP_ID)

# 2. Freeze the Base Model
for param in vision_tower.parameters():
    param.requires_grad = False

# 3. Inject LoRA Adapters
# We target query and value projections in the Transformer
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

# =========================================================
# 4. SETUP SIGLIP VISION TOWER (Corrected)
# =========================================================
print("Loading SigLIP Vision Model...")
processor = AutoProcessor.from_pretrained(SIGLIP_ID)

# FIX: Load ONLY the Vision Encoder
vision_tower = SiglipVisionModel.from_pretrained(SIGLIP_ID)

# A. Freeze Base Model
for param in vision_tower.parameters():
    param.requires_grad = False

# B. Inject LoRA
# The module names are correct for the vision encoder too
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none"
)

vision_tower = get_peft_model(vision_tower, peft_config)
vision_tower.print_trainable_parameters()
vision_tower.to("cuda")


In [ ]:
#Adapation with warm-up

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import SiglipVisionModel, AutoProcessor
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
from PIL import Image
import itertools

# =========================================================
# 1. CONFIGURATION
# =========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16            # Reduce to 8 if you get OOM
WARMUP_EPOCHS = 3          # Train Discriminator only
TOTAL_EPOCHS = 15          # Total training epochs
LR_LORA = 2e-4             # Learning Rate for Vision Adapter
LR_DISC = 2e-4             # Learning Rate for Discriminator
SIGLIP_ID = "google/siglip-so400m-patch14-384"

# =========================================================
# 2. DATASET (Uses Processor)
# =========================================================
class SigLIPDomainDataset(Dataset):
    def __init__(self, csv_path, image_root, domain_label, processor, layout="flat"):
        self.df = pd.read_csv(csv_path)
        self.image_root = image_root
        self.domain_label = domain_label
        self.processor = processor
        self.layout = layout

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = str(row["image_id"])

        # Path Logic
        if self.layout == "flat":
            img_path = os.path.join(self.image_root, image_id + ".jpg")
        else: # Nested
            img_path = os.path.join(self.image_root, image_id, image_id + ".jpg")

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (384, 384), color='black') # Fallback

        # PROCESSOR: Handles Resize (384), Normalize
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs.pixel_values.squeeze(0) # [3, 384, 384]

        domain = torch.tensor(self.domain_label, dtype=torch.float)

        # Fake task label (Replace with actual labels if you have them)
        task_label = torch.zeros(1)

        return pixel_values, domain, task_label

# =========================================================
# 3. GRADIENT REVERSAL LAYER
# =========================================================
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None

def grl(x, lambda_):
    return GradientReversal.apply(x, lambda_)

# =========================================================
# 4. SETUP SIGLIP + LoRA
# =========================================================
print("Loading SigLIP Vision Model...")
processor = AutoProcessor.from_pretrained(SIGLIP_ID)
vision_tower = SiglipVisionModel.from_pretrained(SIGLIP_ID)

# A. Freeze Base Model
for param in vision_tower.parameters():
    param.requires_grad = False

# B. Inject LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none"
)

vision_tower = get_peft_model(vision_tower, peft_config)
vision_tower.to(DEVICE)
print("✅ LoRA Injected.")

# =========================================================
# 5. DISCRIMINATOR
# =========================================================
FEATURE_DIM = vision_tower.config.hidden_size

class DomainDiscriminator(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

discriminator = DomainDiscriminator(FEATURE_DIM).to(DEVICE)
task_classifier = nn.Linear(FEATURE_DIM, 1).to(DEVICE)

# =========================================================
# 6. OPTIMIZERS & DATA LOADERS
# =========================================================
# Optimizer for LoRA + Task Head
optimizer_gen = torch.optim.AdamW(
    list(vision_tower.parameters()) + list(task_classifier.parameters()),
    lr=LR_LORA
)
# Optimizer for Discriminator
optimizer_disc = torch.optim.AdamW(discriminator.parameters(), lr=LR_DISC)

criterion = nn.BCEWithLogitsLoss()

# Loaders
mimic_ds = SigLIPDomainDataset("mimic_standardized.csv", "/content/Seleccted_mimic_CXR_images", 0, processor, "flat")
naija_ds = SigLIPDomainDataset("naija_standardized.csv", "/content/RSFUTH_CXR", 1, processor, "nested")

mimic_loader = DataLoader(mimic_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
naija_loader = DataLoader(naija_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

# =========================================================
# 7. TRAINING LOOP (WITH WARM-UP)
# =========================================================
# ... (Configuration and Loaders remain the same) ...

print(f"Starting Training with {WARMUP_EPOCHS} Warm-Up Epochs (Fixed)...")

for epoch in range(TOTAL_EPOCHS):

    # --- PHASE CHECK ---
    is_warmup = epoch < WARMUP_EPOCHS
    phase_name = "WARM-UP (Disc Only)" if is_warmup else "ADVERSARIAL (Joint)"

    # Set Modes
    discriminator.train()
    if is_warmup:
        vision_tower.eval()
        for p in vision_tower.parameters(): p.requires_grad = False
        for p in discriminator.parameters(): p.requires_grad = True
    else:
        vision_tower.train()
        for n, p in vision_tower.named_parameters():
            if "lora" in n: p.requires_grad = True

    # Dynamic Lambda
    if is_warmup:
        lambda_adv = 0.0
    else:
        p = (epoch - WARMUP_EPOCHS) / (TOTAL_EPOCHS - WARMUP_EPOCHS)
        lambda_adv = 2.0 / (1.0 + np.exp(-10 * p)) - 1.0

    # Dataset Cycling
    if len(mimic_loader) > len(naija_loader):
        loader = zip(mimic_loader, itertools.cycle(naija_loader))
        total_steps = len(mimic_loader)
    else:
        loader = zip(itertools.cycle(mimic_loader), naija_loader)
        total_steps = len(naija_loader)

    loop = tqdm(loader, total=total_steps, desc=f"Ep [{epoch+1}/{TOTAL_EPOCHS}] {phase_name}")

    running_d_loss = 0.0

    for (img_s, d_s, label_s), (img_t, d_t, _) in loop:
        img_s, d_s, label_s = img_s.to(DEVICE), d_s.to(DEVICE), label_s.to(DEVICE)
        img_t, d_t = img_t.to(DEVICE), d_t.to(DEVICE)

        # Zero Gradients for BOTH optimizers at start
        optimizer_gen.zero_grad()
        optimizer_disc.zero_grad()

        imgs_all = torch.cat([img_s, img_t], dim=0)

        # 1. Forward SigLIP
        if is_warmup:
            with torch.no_grad():
                outputs = vision_tower(pixel_values=imgs_all)
                seq_output = outputs.last_hidden_state
        else:
            outputs = vision_tower(pixel_values=imgs_all)
            seq_output = outputs.last_hidden_state

        # 2. Global Pooling
        features_all = seq_output.mean(dim=1)
        feat_s = features_all[:len(img_s)]

        # 3. Calculate Losses

        # A) Domain Loss (Always calculated)
        feat_adv = grl(features_all, lambda_adv)
        d_pred = discriminator(feat_adv)
        d_labels = torch.cat([d_s, d_t], dim=0)
        loss_domain = criterion(d_pred, d_labels)

        # 4. Backward & Step (Branching Logic)
        if is_warmup:
            # Phase 1: Only update Discriminator
            loss_domain.backward()
            optimizer_disc.step()
        else:
            # Phase 2: Joint Update (Single Backward Pass)
            # Calculate Task Loss
            task_pred = task_classifier(feat_s)
            loss_task = criterion(task_pred, label_s)

            # Combine Losses
            # The GRL layer inside 'feat_adv' handles the gradient reversal automatically.
            loss_total = loss_task + loss_domain

            loss_total.backward()

            # Update Both
            optimizer_gen.step()
            optimizer_disc.step()

        running_d_loss += loss_domain.item()
        loop.set_postfix(d_loss=loss_domain.item())

    # Save Checkpoint
    save_path = f"siglip_lora_epoch_{epoch+1}"
    vision_tower.save_pretrained(save_path)
    print(f"Epoch {epoch+1} Avg D-Loss: {running_d_loss/total_steps:.4f}")


In [ ]:
#Saving Adapted encoder to drive
from google.colab import drive
import shutil
import os

# 2. Define Source and Destination
source_folder = "siglip_lora_epoch_15"  # The folder in Colab
destination_folder = "/content/drive/MyDrive/NaijaCXR_Project/siglip_lora_epoch_15" # Folder in Drive

# 3. Copy the folder
if os.path.exists(destination_folder):
    print(f"Warning: {destination_folder} already exists. Removing old version...")
    shutil.rmtree(destination_folder)

print(f"Copying to Google Drive...")
shutil.copytree(source_folder, destination_folder)
print(f"✅ Success! Adapter saved to: {destination_folder}")


## 5. Domain Invariance Verification (t-SNE & Probes)


In [ ]:
#PLotting tsne to ensure the vision encoder is adapted to NairaCXR.

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from transformers import SiglipVisionModel, AutoProcessor
from peft import PeftModel, PeftConfig
from PIL import Image
import os
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL_ID = "google/siglip-so400m-patch14-384"

# Path to your saved adapter folder (Change this to your best epoch!)
ADAPTER_PATH = "/content/drive/MyDrive/NaijaCXR_Project/siglip_lora_epoch_15"

# Data Paths
MIMIC_CSV = "mimic_standardized.csv"
MIMIC_ROOT = "/content/Seleccted_mimic_CXR_images"
NAIJA_CSV = "naija_standardized.csv"
NAIJA_ROOT = "/content/RSFUTH_CXR"

# ==========================================
# 2. DATASET CLASS (Simplified for Inference)
# ==========================================
class InferenceDataset(Dataset):
    def __init__(self, csv_path, image_root, processor, layout="flat", limit=None):
        self.df = pd.read_csv(csv_path)
        if limit:
            self.df = self.df.sample(n=min(limit, len(self.df)), random_state=42)
        self.image_root = image_root
        self.processor = processor
        self.layout = layout

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = str(row["image_id"])

        if self.layout == "flat":
            img_path = os.path.join(self.image_root, image_id + ".jpg")
        else:
            img_path = os.path.join(self.image_root, image_id, image_id + ".jpg")

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (384, 384), color='black')

        inputs = self.processor(images=image, return_tensors="pt")
        return inputs.pixel_values.squeeze(0)

# ==========================================
# 3. HELPER: FEATURE EXTRACTION
# ==========================================
def extract_features(model, loader, desc="Extracting"):
    model.eval()
    features = []

    with torch.no_grad():
        for imgs in tqdm(loader, desc=desc):
            imgs = imgs.to(DEVICE)

            # Forward Pass
            outputs = model(pixel_values=imgs)

            # Global Average Pooling [Batch, Patches, Dim] -> [Batch, Dim]
            # We take the mean of all patch tokens to represent the image
            feats = outputs.last_hidden_state.mean(dim=1)

            features.append(feats.cpu().numpy())

    return np.vstack(features)

# ==========================================
# 4. LOAD MODELS
# ==========================================
print("Loading Models...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

# --- A. Load Baseline (Before Adaptation) ---
# Pure SigLIP without adapters
model_baseline = SiglipVisionModel.from_pretrained(BASE_MODEL_ID)
model_baseline.to(DEVICE)

# --- B. Load Adapted (After Adaptation) ---
# Base SigLIP + Your LoRA Adapters
model_adapted = SiglipVisionModel.from_pretrained(BASE_MODEL_ID)
model_adapted = PeftModel.from_pretrained(model_adapted, ADAPTER_PATH)
model_adapted.to(DEVICE)

print("Models Loaded!")

# ==========================================
# 5. EXTRACT FEATURES
# ==========================================
SAMPLES = 600 # Number of images per domain to plot (keep it balanced)

# Create Loaders
mimic_ds = InferenceDataset(MIMIC_CSV, MIMIC_ROOT, processor, "flat", limit=SAMPLES)
naija_ds = InferenceDataset(NAIJA_CSV, NAIJA_ROOT, processor, "nested", limit=SAMPLES)

mimic_loader = DataLoader(mimic_ds, batch_size=32, shuffle=False)
naija_loader = DataLoader(naija_ds, batch_size=32, shuffle=False)

# Extract for Baseline
print("\n--- Processing Baseline (Before) ---")
mimic_feats_b = extract_features(model_baseline, mimic_loader, "MIMIC Baseline")
naija_feats_b = extract_features(model_baseline, naija_loader, "Naija Baseline")

# Extract for Adapted
print("\n--- Processing Adapted (After) ---")
mimic_feats_a = extract_features(model_adapted, mimic_loader, "MIMIC Adapted")
naija_feats_a = extract_features(model_adapted, naija_loader, "Naija Adapted")

# ==========================================
# 6. COMPUTE t-SNE AND PLOT
# ==========================================
def plot_tsne(mimic_f, naija_f, title, ax):
    # Combine
    X = np.vstack([mimic_f, naija_f])
    y = np.hstack([np.zeros(len(mimic_f)), np.ones(len(naija_f))]) # 0=MIMIC, 1=Naija

    # Run t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='pca', learning_rate='auto')
    X_embedded = tsne.fit_transform(X)

    # Plot
    df = pd.DataFrame({
        "x": X_embedded[:, 0], "y": X_embedded[:, 1],
        "Domain": ["MIMIC (Source)" if label==0 else "Naija (Target)" for label in y]
    })

    sns.scatterplot(
        data=df, x="x", y="y", hue="Domain", style="Domain",
        alpha=0.6, s=20, ax=ax,
        palette={'MIMIC (Source)': 'blue', 'Naija (Target)': 'orange'}
    )
    ax.set_title(title)
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.3)

# Generate Figure
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

plot_tsne(mimic_feats_b, naija_feats_b, "Before Adaptation (SigLIP Baseline)", axes[1])
plot_tsne(mimic_feats_a, naija_feats_a, "After Adaptation (SigLIP + LoRA)", axes[0])

plt.tight_layout()
plt.show()

# =========================================================
# 7. QUANTITATIVE DOMAIN-SEPARATION CHECK (NEW)
# =========================================================
# Visual inspection of t-SNE plots can be misleading. Here we fit a simple
# linear classifier (domain discriminator proxy) on RAW feature space (not
# the 2D t-SNE projection) to check whether MIMIC vs NaijaCXR features are
# still trivially separable before adaptation, and whether adaptation
# changed that. If "After Adaptation" accuracy is still ~1.0 (or unchanged
# from baseline), the SigLIP LoRA adapter did NOT meaningfully shift
# NaijaCXR features toward MIMIC's manifold -- the LLM will still receive
# MIMIC-like visual representations for NaijaCXR images, which can bias
# generated reports toward MIMIC's "normal study" style.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

def domain_separability(mimic_f, naija_f, label=""):
    X = np.vstack([mimic_f, naija_f])
    y = np.hstack([np.zeros(len(mimic_f)), np.ones(len(naija_f))])
    clf = LogisticRegression(max_iter=1000)
    scores = cross_val_score(clf, X, y, cv=5)
    print(f"{label} domain-classifier accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")
    return scores.mean()

print("\n\U0001F50D Quantitative Domain Separability:")
acc_before = domain_separability(mimic_feats_b, naija_feats_b, "Before adaptation")
acc_after = domain_separability(mimic_feats_a, naija_feats_a, "After adaptation ")

if acc_after > 0.85 and abs(acc_before - acc_after) < 0.05:
    print("\n\u26A0\uFE0F  WARNING: Domains remain highly separable after adaptation, "
          "and adaptation barely changed separability. The vision encoder may not "
          "have meaningfully adapted to NaijaCXR. Consider: more adaptation epochs, "
          "a higher lambda_adv schedule, or a larger LoRA rank for the SigLIP tower.")
elif acc_after < acc_before:
    print(f"\n\u2705 Adaptation reduced domain separability ({acc_before:.3f} -> {acc_after:.3f}), "
          "consistent with successful domain-invariant feature learning.")


## 6. MedGemma-4B Model Architecture & Surgery
We load base **MedGemma-4B-IT**, transplant the domain-adapted SigLIP vision tower, resize the positional embeddings to 729 tokens, and patch the multi-modal projector.


In [ ]:
#New fixes
import torch
import torch.nn.functional as F
import types
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    SiglipVisionModel
)
from peft import LoraConfig, get_peft_model, PeftModel
from PIL import Image
import numpy as np

# =========================================================
# 1. CONFIGURATION
# =========================================================
MODEL_ID = "google/medgemma-4b-it"
SIGLIP_ID = "google/siglip-so400m-patch14-384"
ADAPTER_PATH = "/content/drive/MyDrive/NaijaCXR_Project/siglip_lora_epoch_15"

# Target Settings for your Custom Encoder
TARGET_IMAGE_SIZE = 384
TARGET_PATCH_SIZE = 14
# Math: (384/14)^2 = 27*27 = 729 tokens
TARGET_TOKENS = 729
OLD_TOKENS = 256 # Default MedGemma tokens

print("🏥 Starting Fresh Initialization...")

# =========================================================
# 2. LOAD PROCESSOR & FORCE CONFIG
# =========================================================
print("⚙️  Configuring Processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
# Force processor to respect 384px
processor.image_processor.size = {"height": TARGET_IMAGE_SIZE, "width": TARGET_IMAGE_SIZE}
processor.image_processor.crop_size = {"height": TARGET_IMAGE_SIZE, "width": TARGET_IMAGE_SIZE}
processor.image_processor.do_resize = True

# =========================================================
# 3. LOAD MODEL (LLM)
# =========================================================
print(f"📥 Loading {MODEL_ID} in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# =========================================================
# 4. TRANSPLANT VISION TOWER
# =========================================================
print("🔌 Transplanting Custom SigLIP Tower...")
clean_tower = SiglipVisionModel.from_pretrained(SIGLIP_ID, torch_dtype=torch.bfloat16)
clean_tower = PeftModel.from_pretrained(clean_tower, ADAPTER_PATH)
clean_tower = clean_tower.merge_and_unload()
clean_tower.to("cuda")

# Swap it in
if hasattr(model, "vision_tower"):
    model.vision_tower = clean_tower
elif hasattr(model.model, "vision_tower"):
    model.model.vision_tower = clean_tower

# =========================================================
# 5. THE "SURGERY" (Fixing Architectures)
# =========================================================
print("🔪 Performing Model Surgery...")

# A. Patch Configs
model.config.vision_config.image_size = TARGET_IMAGE_SIZE
model.config.vision_config.num_image_tokens = TARGET_TOKENS
model.config.mm_tokens_per_image = TARGET_TOKENS

# B. Resize Position Embeddings (Fixes the 4096 vs 729 crash)
# We find any parameter with size 4096 (64x64) and shrink it to 729 (27x27)
for name, param in model.named_parameters():
    if 4096 in param.shape:
        print(f"   └── Resizing Embedding: {name}")

        # Reshape to [Batch, Dim, 64, 64]
        if param.ndim == 3: # [1, 4096, dim]
            t = param.transpose(1, 2).view(1, -1, 64, 64)
            new_t = F.interpolate(t.float(), size=(27, 27), mode='bicubic')
            new_param = new_t.flatten(2).transpose(1, 2)
        else: # [4096, dim]
            t = param.T.unsqueeze(0).view(1, -1, 64, 64)
            new_t = F.interpolate(t.float(), size=(27, 27), mode='bicubic')
            new_param = new_t.flatten(2).transpose(1, 2).squeeze(0)

        param.data = new_param.to(param.dtype).to(param.device)

# C. Unfreeze & Fix Projector
# Find the projector
projector = model.model.multi_modal_projector if hasattr(model.model, "multi_modal_projector") else model.multi_modal_projector

# De-quantize projector for training
print("   └── Unfreezing Projector...")
projector.to(dtype=torch.bfloat16)
projector.train()

# Remove the incompatible Pooling Layer
if hasattr(projector, "avg_pool"):
    projector.avg_pool = torch.nn.Identity()

# Force update internal attribute 'patches_per_image'
for m in projector.modules():
    if hasattr(m, "patches_per_image"):
        m.patches_per_image = 27

# =========================================================
# 6. SETUP LORA
# =========================================================
print("🚀 Configuring LoRA...")
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["multi_modal_projector"], # TRAIN PROJECTOR FULLY
    task_type="CAUSAL_LM",
    lora_dropout=0.05
)
model = get_peft_model(model, lora_config)

# =========================================================
# 7. INPUT HELPER (The Final Piece)
# =========================================================
# This function guarantees inputs match the 729 token requirement
def prepare_inputs(text, image, processor, device="cuda"):
    # 1. Standard Process
    inputs = processor(text=text, images=image, return_tensors="pt").to(device)

    # 2. Force Pixel Size (384)
    if inputs["pixel_values"].shape[2] != TARGET_IMAGE_SIZE:
        inputs["pixel_values"] = F.interpolate(
            inputs["pixel_values"], size=(TARGET_IMAGE_SIZE, TARGET_IMAGE_SIZE),
            mode='bilinear', align_corners=False
        )
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

    # 3. Fix Token Count (256 -> 729)
    input_ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    has_token_types = "token_type_ids" in inputs
    token_types = inputs.get("token_type_ids")

    # Find the sequence of 256 tokens and replace with 729
    new_ids_list = []
    new_mask_list = []
    new_types_list = []

    for i in range(len(input_ids)):
        seq = input_ids[i].tolist()

        # Simple Logic: Find the image token ID
        u, c = torch.unique(input_ids[i], return_counts=True)
        img_id = u[c == OLD_TOKENS]
        if len(img_id) > 0:
            img_id = img_id[0].item()
            start = seq.index(img_id)

            # Reconstruct
            prefix = seq[:start]
            img_block = [img_id] * TARGET_TOKENS
            suffix = seq[start + OLD_TOKENS:]

            new_ids_list.append(prefix + img_block + suffix)
            new_mask_list.append([1]*len(prefix) + [1]*TARGET_TOKENS + [1]*len(suffix))

            if has_token_types:
                new_types_list.append([0]*len(prefix) + [1]*TARGET_TOKENS + [0]*len(suffix))
        else:
            # Fallback if detection fails
            new_ids_list.append(seq)
            new_mask_list.append(mask[i].tolist())
            if has_token_types:
                new_types_list.append(token_types[i].tolist())

    inputs["input_ids"] = torch.tensor(new_ids_list, device=device)
    inputs["attention_mask"] = torch.tensor(new_mask_list, device=device)
    if has_token_types:
        inputs["token_type_ids"] = torch.tensor(new_types_list, device=device)

    return inputs

# =========================================================
# 8. VERIFY
# =========================================================
print("\n🧪 Running Final Verification...")
dummy_img = Image.fromarray(np.zeros((384, 384, 3), dtype=np.uint8))
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe this x-ray"}]}]
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

final_inputs = prepare_inputs(prompt, dummy_img, processor)

with torch.no_grad():
    out = model(**final_inputs)
    print("✅ System Operational.")
    print(f"   Output Logits: {out.logits.shape}")


## 7. Supervised Fine-Tuning (SFT) on NaijaCXR


In [ ]:
import json
import os

# Configuration
FILES_TO_CLEAN = ["mimic_train.json", "Naija_train.json"]
CLEANED_SUFFIX = "_cleaned.json"

def clean_dataset(filename):
    print(f"🧹 Cleaning {filename}...")

    with open(filename, 'r') as f:
        data = json.load(f)

    clean_data = []
    removed_nan_count = 0
    removed_missing_file_count = 0

    for item in data:
        # 1. Extract the Assistant's response (Ground Truth)
        messages = item.get("messages", [])
        assistant_text = ""
        user_image_path = ""

        for msg in messages:
            if msg["role"] == "assistant":
                assistant_text = msg["content"][0]["text"]
            if msg["role"] == "user":
                # Find image path
                for content in msg["content"]:
                    if content["type"] == "image":
                        user_image_path = content["image"]

        # 2. Check for "nan" (Case insensitive)
        # We look for "nan" appearing as a distinct word or line
        if "\nnan" in assistant_text.lower() or " nan " in assistant_text.lower() or assistant_text.lower().strip() == "nan":
            removed_nan_count += 1
            continue # SKIP this bad sample

        # 3. Check if image file exists (Optional but recommended)
        # Handle MIMIC paths relative to /content/
        full_path = user_image_path
        if "Seleccted_mimic_CXR_images" in user_image_path and not user_image_path.startswith("/"):
             full_path = os.path.join("/content", user_image_path)

        if not os.path.exists(full_path):
            # Uncomment the line below to strictly remove missing files
            # removed_missing_file_count += 1
            # continue
            pass # For now, just warning

        # If it passes checks, keep it
        clean_data.append(item)

    # Save cleaned file
    new_filename = filename.replace(".json", CLEANED_SUFFIX)
    with open(new_filename, 'w') as f:
        json.dump(clean_data, f, indent=2)

    print(f"   Original size: {len(data)}")
    print(f"   Removed 'nan': {removed_nan_count}")
    print(f"   Removed missing files: {removed_missing_file_count}")
    print(f"   ✅ Saved {len(clean_data)} valid samples to {new_filename}\n")
    return new_filename

# Run cleaning
cleaned_files = []
for f_name in FILES_TO_CLEAN:
    if os.path.exists(f_name):
        cleaned_files.append(clean_dataset(f_name))
    else:
        print(f"⚠️ File not found: {f_name}")

print(" 👇 USE THESE FILES FOR TRAINING NOW:")
print(cleaned_files)

import os
from datasets import load_dataset
from PIL import Image

# =========================================================
# 1. LOAD DATASET (FAST)
# =========================================================
print("\U0001F4C2 Loading JSONs...")
data_files = {
    "train": ["Naija_train_cleaned.json", "mimic_train_cleaned.json"]
}

# Keep 'keep_in_memory=False' to save RAM
dataset = load_dataset("json", data_files=data_files, split="train", keep_in_memory=False)
print(f"\u2705 Loaded {len(dataset)} examples.")

# =========================================================
# 1b. DOMAIN BALANCE DIAGNOSTIC (see also the rebalancing cell below)
# =========================================================
def _is_mimic(example):
    try:
        p = example["messages"][0]["content"][0]["image"]
        return "Seleccted_mimic_CXR_images" in p
    except Exception:
        return False

n_mimic = sum(1 for ex in dataset if _is_mimic(ex))
n_naija = len(dataset) - n_mimic
print(f"\U0001F4CA Domain counts -> MIMIC: {n_mimic} | NaijaCXR: {n_naija}")
if n_mimic > 0 and n_naija > 0:
    ratio = n_mimic / n_naija
    print(f"   MIMIC:NaijaCXR ratio = {ratio:.2f} : 1")
    if ratio > 1.5 or ratio < 0.67:
        print("   \u26A0\uFE0F  Significant domain imbalance detected. "
              "Run the rebalancing cell below before applying the lazy transform.")

# =========================================================
# 2. PATH VERIFICATION (Safety Check)
# =========================================================
print("\n\U0001F575\uFE0F  Verifying Paths...")
def check_path_exists(example):
    try:
        raw_path = example["messages"][0]["content"][0]["image"]
        if "Seleccted_mimic_CXR_images" in raw_path and not raw_path.startswith("/"):
            path = os.path.join("/content", raw_path)
        else:
            path = raw_path

        if os.path.exists(path):
            return True
        else:
            return False
    except:
        return False

for i in [0, 1, len(dataset)-2, len(dataset)-1]:
    exists = check_path_exists(dataset[i])
    path_sample = dataset[i]["messages"][0]["content"][0]["image"]
    status = "\u2705 Found" if exists else "\u274C NOT FOUND"
    print(f"   [{i}] {status}: {path_sample}")


# =========================================================
# OPTIONAL BUT RECOMMENDED: Domain Balancing
# =========================================================
# MIMIC-CXR contains a large proportion of "normal" reports. If MIMIC
# examples outnumber NaijaCXR examples, the model's report-style prior
# will be dominated by MIMIC's distribution, biasing generations toward
# generic "normal study" text even for NaijaCXR (target-domain) images.
#
# This cell upsamples NaijaCXR examples (via repetition with re-shuffling)
# so that, per epoch, the Trainer sees roughly equal numbers of MIMIC and
# NaijaCXR examples. Run this AFTER loading `dataset` (previous cell) and
# BEFORE calling `dataset.with_transform(lazy_processor)`.

import random

def _is_mimic_idx(example):
    try:
        p = example["messages"][0]["content"][0]["image"]
        return "Seleccted_mimic_CXR_images" in p
    except Exception:
        return False

mimic_indices = [i for i in range(len(dataset)) if _is_mimic_idx(dataset[i])]
naija_indices = [i for i in range(len(dataset)) if i not in set(mimic_indices)]

print(f"MIMIC examples: {len(mimic_indices)} | NaijaCXR examples: {len(naija_indices)}")

if len(naija_indices) > 0 and len(mimic_indices) > 0:
    target_n = max(len(mimic_indices), len(naija_indices))

    rng = random.Random(42)

    def _upsample(indices, target):
        if len(indices) >= target:
            return indices
        reps = (target // len(indices)) + 1
        pool = indices * reps
        rng.shuffle(pool)
        return pool[:target]

    balanced_mimic = _upsample(mimic_indices, target_n)
    balanced_naija = _upsample(naija_indices, target_n)

    balanced_indices = balanced_mimic + balanced_naija
    rng.shuffle(balanced_indices)

    dataset = dataset.select(balanced_indices)
    print(f"\u2705 Rebalanced dataset size: {len(dataset)} "
          f"(MIMIC: {len(balanced_mimic)}, NaijaCXR: {len(balanced_naija)})")
else:
    print("\u26A0\uFE0F  Could not detect both domains -- skipping rebalancing.")


# =========================================================
# 3. LAZY TRANSFORM (FIXED: prefix-aligned prompt/full text)
# =========================================================
# CRITICAL FIX:
# Previously, `formatted_text` (full 2-turn conversation) and
# `prompt_text` (user turn only, tokenize=False vs add_generation_prompt=True)
# were generated via TWO INDEPENDENT calls to apply_chat_template.
# These can diverge in tokenization at the turn boundary (Gemma's
# <start_of_turn>model marker is only added when add_generation_prompt=True),
# so `prompt_inputs["input_ids"].shape[1]` did NOT correspond to the true
# prefix length inside `inputs["input_ids"]`. This misaligned the label
# masking in the collator: `labels[:prompt_len] = -100` masked an incorrect
# span, often wiping out the assistant's report tokens from the training
# signal entirely. As a result, the model received almost no supervision on
# how to actually WRITE a report, and collapsed to MedGemma's prior
# ("no acute cardiopulmonary process" / normal study) at inference.
#
# FIX: Build `prompt_text` first (with add_generation_prompt=True), then
# construct `formatted_text` by STRING CONCATENATION:
#     formatted_text = prompt_text + assistant_response + "<end_of_turn>\n"
# This guarantees `prompt_text` is an exact prefix of `formatted_text`,
# so after the 256->729 image-token expansion (which only affects the
# image-token block shared by both), prompt_inputs["input_ids"] remains
# an exact prefix of inputs["input_ids"]. Label masking is now correct.
def lazy_processor(batch):
    # 'batch' is a dictionary of lists: {'messages': [[...], [...]]}
    formatted_texts = []
    prompt_texts = []
    pil_images = []

    eos_token = processor.tokenizer.eos_token or "<end_of_turn>"

    for messages in batch["messages"]:
        # A. EXTRACT PATH
        user_msg = messages[0]
        raw_path = None
        for item in user_msg["content"]:
            if item["type"] == "image":
                raw_path = item["image"]
                break

        # B. RESOLVE PATH
        if raw_path:
            if "Seleccted_mimic_CXR_images" in raw_path and not raw_path.startswith("/"):
                image_path = os.path.join("/content", raw_path)
            else:
                image_path = raw_path # Naija paths are usually absolute
        else:
            image_path = None

        # C. OPEN IMAGE
        try:
            if image_path and os.path.exists(image_path):
                img = Image.open(image_path).convert("RGB")
            else:
                # Create a black dummy image if missing (prevents crash)
                img = Image.new('RGB', (384, 384), (0, 0, 0))
        except Exception:
            img = Image.new('RGB', (384, 384), (0, 0, 0))

        pil_images.append(img)

        # D. CLEAN MESSAGES (strip image content down to {"type": "image"})
        clean_msgs = []
        for msg in messages:
            content_list = []
            for item in msg["content"]:
                if item["type"] == "text":
                    content_list.append(item)
                elif item["type"] == "image":
                    content_list.append({"type": "image"})
            clean_msgs.append({"role": msg["role"], "content": content_list})

        # E. BUILD PROMPT TEXT (user turn + generation marker)
        prompt_msgs = [clean_msgs[0]]
        prompt_text = processor.apply_chat_template(prompt_msgs, add_generation_prompt=True)
        prompt_texts.append(prompt_text)

        # F. EXTRACT ASSISTANT RESPONSE (raw text)
        assistant_text = ""
        for msg in clean_msgs[1:]:
            if msg["role"] == "assistant":
                for c in msg["content"]:
                    if c["type"] == "text":
                        assistant_text += c["text"]

        # G. BUILD FULL TEXT AS prompt_text + response (PREFIX-SAFE)
        full_text = prompt_text + assistant_text + eos_token + "\n"
        formatted_texts.append(full_text)

    return {"formatted_text": formatted_texts, "prompt_text": prompt_texts, "image": pil_images}

# APPLY THE LAZY TRANSFORM
# This is instant because it doesn't process data yet
train_dataset = dataset.with_transform(lazy_processor)

print("\n\U0001F680 Dataset ready for Trainer (Lazy Mode Enabled, Label Alignment Fixed)")


# =========================================================
# 1. FIXED COLLATOR (Prefix-aligned label masking + token type padding)
# =========================================================
from torch.nn.utils.rnn import pad_sequence
import torch

class NaijaCXRCollator:
    def __init__(self, processor):
        self.processor = processor
        if self.processor.tokenizer.pad_token_id is None:
            self.processor.tokenizer.pad_token_id = self.processor.tokenizer.eos_token_id

    def __call__(self, examples):
        texts = [x['formatted_text'] for x in examples]
        prompts = [x['prompt_text'] for x in examples]
        images = [x['image'] for x in examples]

        batch_input_ids = []
        batch_attention_mask = []
        batch_pixel_values = []
        batch_token_types = []
        batch_labels = []

        for i in range(len(texts)):
            # Use CPU to avoid GPU frag
            inputs = prepare_inputs(texts[i], images[i], self.processor, device="cpu")
            prompt_inputs = prepare_inputs(prompts[i], images[i], self.processor, device="cpu")

            input_ids = inputs["input_ids"][0]
            attention_mask = inputs["attention_mask"][0]
            pixel_values = inputs["pixel_values"][0]
            token_type_ids = inputs["token_type_ids"][0]

            # Create labels and mask the prompt tokens (setting to -100)
            labels = input_ids.clone()
            prompt_len = prompt_inputs["input_ids"].shape[1]

            # SAFETY CHECK: prompt_inputs should be an exact prefix of inputs.
            # Because lazy_processor now builds formatted_text = prompt_text + response,
            # and prepare_inputs applies the SAME 256->729 image-token expansion to
            # the shared prefix, this should always hold. If it doesn't (e.g. due to
            # an edge case in tokenization), fall back to NOT masking anything rather
            # than silently masking the wrong (possibly entire) span.
            full_len = input_ids.shape[0]
            if prompt_len >= full_len:
                # Degenerate case: prompt consumed the whole (or more than the) sequence.
                # Don't mask everything -- mask nothing so this sample still contributes
                # signal, but flag it.
                prompt_len = 0
            elif not torch.equal(input_ids[:prompt_len], prompt_inputs["input_ids"][0][:prompt_len]):
                # Prefix mismatch -- skip masking for this sample to avoid wiping out
                # the response tokens. (Should be rare/never with the fixed pipeline.)
                prompt_len = 0

            labels[:prompt_len] = -100

            batch_input_ids.append(input_ids)
            batch_attention_mask.append(attention_mask)
            batch_pixel_values.append(pixel_values)
            batch_token_types.append(token_type_ids)
            batch_labels.append(labels)

        # PADDING
        input_ids = pad_sequence(batch_input_ids, batch_first=True, padding_value=self.processor.tokenizer.pad_token_id)
        attention_mask = pad_sequence(batch_attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(batch_labels, batch_first=True, padding_value=-100)

        # Pad token types with 0 (Text)
        token_type_ids = pad_sequence(batch_token_types, batch_first=True, padding_value=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": torch.stack(batch_pixel_values),
            "token_type_ids": token_type_ids,
            "labels": labels
        }


# =========================================================
# QUICK SANITY CHECK (run before training)
# =========================================================
# Verify that, for a real example, the label mask leaves a non-trivial
# number of supervised (non -100) tokens corresponding to the report text.
_collator = NaijaCXRCollator(processor)
_sample_batch = _collator([train_dataset[0], train_dataset[1]])
for _i in range(2):
    _n_supervised = (_sample_batch["labels"][_i] != -100).sum().item()
    _total = _sample_batch["labels"][_i].shape[0]
    print(f"Sample {_i}: {_n_supervised}/{_total} tokens supervised (should be > 0 and roughly equal to the report length).")
    if _n_supervised == 0:
        print("   \u26A0\uFE0F  WARNING: zero supervised tokens -- label masking is still broken!")


# =========================================================
# 2. START TRAINING
# =========================================================
from transformers import Trainer, TrainingArguments

# Sanity check: confirm trainable parameters exist (LoRA + projector)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\U0001F50E Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
if trainable == 0:
    raise RuntimeError("No trainable parameters! Check LoRA/get_peft_model setup before training.")

training_args = TrainingArguments(
    output_dir="./medgemma-naijacxr-final",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=1e-4,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    report_to="none",
    # Ensure the Trainer treats 'labels' correctly for the PEFT-wrapped
    # CausalLM (required in some transformers/peft version combos, otherwise
    # the loss may be computed on the wrong field or not at all).
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    args=training_args, # Uses your existing args
    train_dataset=train_dataset,
    data_collator=NaijaCXRCollator(processor),
)

print("\U0001F1F3\U0001F1EC Starting Training (Label Alignment Fixed)...")
trainer.train()

# Quick post-training check: loss should have decreased meaningfully from
# the first logged step. If it barely moved, the model likely did not learn
# anything useful and predictions will still default to generic reports.
try:
    log_history = trainer.state.log_history
    losses = [l["loss"] for l in log_history if "loss" in l]
    if len(losses) >= 2:
        print(f"\U0001F4C9 First logged loss: {losses[0]:.4f} -> Last logged loss: {losses[-1]:.4f}")
        if losses[0] - losses[-1] < 0.05:
            print("   \u26A0\uFE0F  Loss barely decreased. Consider: more epochs, higher LR, "
                  "checking the label-alignment sanity check above, or verifying the vision "
                  "tower adaptation (t-SNE plot) actually separated the two domains.")
except Exception as e:
    print(f"(Could not summarize loss history: {e})")


## 8. Sample Generation & Inference Test


In [ ]:
import torch
from PIL import Image

# =========================================================
# 1. DEFINE YOUR NEW PROMPT
# =========================================================
EVAL_PROMPT = "Describe the chest X-ray findings and impression."

# =========================================================
# 2. CUSTOM GENERATION FUNCTION
# =========================================================
def generate_report(model, processor, image_path, prompt=EVAL_PROMPT):
    """
    Generates a report using the "Surgically Altered" MedGemma model.
    Defaults to the new detailed evaluation prompt.
    """
    model.eval()

    # A. Load Image
    if isinstance(image_path, str):
        image = Image.open(image_path).convert("RGB")
    else:
        image = image_path # Allow passing PIL object directly

    # B. Format Prompt with Chat Template
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]
    # We use the processor to handle the chat structure (including <start_of_turn> tokens)
    text_input = processor.apply_chat_template(messages, add_generation_prompt=True)

    # C. PREPARE INPUTS (The Surgery)
    # This calls your custom 'prepare_inputs' helper (defined in your notebook)
    # It handles resizing (384px) and token expansion (256 -> 729)
    inputs = prepare_inputs(text_input, image, processor, device="cuda")

    # D. GENERATE
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            pixel_values=inputs["pixel_values"],
            token_type_ids=inputs["token_type_ids"], # Critical for the hybrid model
            max_new_tokens=200,    # Increased slightly to accommodate "Findings" + "Impression"
            do_sample=True,
            temperature=0.2,       # Low temp for clinical accuracy
            top_p=0.9,
            repetition_penalty=1.1 # Helps prevent repeating phrases
        )

    # E. DECODE
    # Decode only the newly generated tokens
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    report = processor.decode(generated_ids, skip_special_tokens=True)

    return report

# =========================================================
# 3. TEST IT
# =========================================================
# Pick an image from your dataset
test_image_path = "/content/RSFUTH_CXR/P00001/P00001.jpg"

print(f"⏳ Generating Report for {test_image_path}...")
print(f"📝 Using Prompt: {EVAL_PROMPT}\n")

try:
    report = generate_report(model, processor, test_image_path)

    print("\n🩻 RADIOLOGY REPORT:")
    print("=" * 40)
    print(report)
    print("=" * 40)

except Exception as e:
    print(f"❌ Error: {e}")
    print("Tip: Ensure 'prepare_inputs' is defined and the model is loaded.")


## 9. Checkpoint & Configuration Export to Google Drive


In [ ]:
import os
import json
from google.colab import drive

# 1. MOUNT DRIVE
if not os.path.exists("/content/drive"):
    print("🔌 Mounting Google Drive...")
    drive.mount('/content/drive')

# 2. DEFINE OUTPUT PATH
DRIVE_FOLDER = "/content/drive/MyDrive/NaijaCXR_Project"
OUTPUT_DIR = os.path.join(DRIVE_FOLDER, "medgemma_naijacxr_V2")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"📁 Created directory: {OUTPUT_DIR}")

print(f"💾 Saving artifacts to: {OUTPUT_DIR} ...")

# 3. SAVE MODEL & ADAPTERS
trainer.save_model(OUTPUT_DIR)
print("   ✅ Model weights (Adapters + Projector) saved.")

# ---------------------------------------------------------
# 3.b. FORCE SAVE BASE CONFIG (The Fix for FileNotFoundError)
# ---------------------------------------------------------
# LoRA models don't save 'config.json' by default. We must trigger it manually.
print("   💾 Saving Base Configuration...")
model.config.save_pretrained(OUTPUT_DIR)

# 4. SAVE PROCESSOR
processor.save_pretrained(OUTPUT_DIR)
print("   ✅ Processor saved.")

# 5. MANUAL JSON PATCHING (Now safe to run)
print("   🔧 Patching Configuration manually...")
config_path = os.path.join(OUTPUT_DIR, "config.json")

# A. Read the file (It definitely exists now!)
with open(config_path, 'r') as f:
    config_data = json.load(f)

# B. Inject VLM settings
if "vision_config" not in config_data:
    config_data["vision_config"] = {}

config_data["vision_config"]["num_image_tokens"] = 729
config_data["vision_config"]["image_size"] = 384
config_data["vision_config"]["patch_size"] = 14
config_data["mm_tokens_per_image"] = 729

# C. Write back
with open(config_path, 'w') as f:
    json.dump(config_data, f, indent=2)

print("   ✅ Patched Configuration (729 tokens) saved.")

# 6. SAVE LOGS
history_path = os.path.join(OUTPUT_DIR, "training_logs.json")
with open(history_path, "w") as f:
    json.dump(trainer.state.log_history, f)
print("   ✅ Training logs saved.")

print("\n🎉 ALL FILES SECURED ON GOOGLE DRIVE!")


## 10. Quantitative Evaluation on Test Splits


In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
import json
import os
import pandas as pd
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, SiglipVisionModel
from peft import PeftModel


# =========================================================
# 1. CONFIGURATION
# =========================================================
BASE_MODEL_ID = "google/medgemma-4b-it"
SIGLIP_ID = "google/siglip-so400m-patch14-384"
# Ensure these paths match your Drive structure
TRAINED_MODEL_PATH = "/content/drive/MyDrive/NaijaCXR_Project/medgemma_naijacxr_V2"
VISION_ADAPTER_PATH = "/content/drive/MyDrive/NaijaCXR_Project/siglip_lora_epoch_15"
TEST_FILE_PATH = "/content/Naija_test.json"
CSV_OUTPUT_PATH = "NaijaCXR-VLM_predictions.csv"

# Model Constants
TARGET_IMG_SIZE = 384
TARGET_TOKENS = 729

# =========================================================
# 2. CORRECTED MODEL LOADER
# =========================================================
def load_evaluation_model():
    print("🏗️  Re-assembling Model for Evaluation...")

    # A. Base Load
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
    )
    model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
    )

    # B. Swap Vision Tower
    print("   Swapping Vision Tower...")
    vision_tower = SiglipVisionModel.from_pretrained(SIGLIP_ID, torch_dtype=torch.bfloat16)
    vision_tower = PeftModel.from_pretrained(vision_tower, VISION_ADAPTER_PATH)
    vision_tower = vision_tower.merge_and_unload()
    vision_tower.to("cuda")
    if hasattr(model, "vision_tower"): model.vision_tower = vision_tower
    elif hasattr(model.model, "vision_tower"): model.model.vision_tower = vision_tower

    # C. Resize Embeddings (4096 -> 729)
    print("   Resizing Embeddings...")
    model.config.mm_tokens_per_image = TARGET_TOKENS
    # FIX: also patch vision_config to match training-time surgery (Cell 15).
    # Without these, the model's own config disagrees with the actual tensor
    # shapes / prepare_inputs token expansion, which can cause silent
    # mismatches between training and evaluation.
    model.config.vision_config.image_size = TARGET_IMG_SIZE
    model.config.vision_config.num_image_tokens = TARGET_TOKENS
    for name, param in model.named_parameters():
        if 4096 in param.shape:
            if param.ndim == 3: t = param.transpose(1, 2).view(1, -1, 64, 64)
            else: t = param.T.unsqueeze(0).view(1, -1, 64, 64)
            new_t = F.interpolate(t.float(), size=(27, 27), mode='bicubic')
            if param.ndim == 3: new_param = new_t.flatten(2).transpose(1, 2)
            else: new_param = new_t.flatten(2).transpose(1, 2).squeeze(0)
            param.data = new_param.to(param.dtype).to(param.device)

    # D. Patch Projector
    projector = model.model.multi_modal_projector if hasattr(model.model, "multi_modal_projector") else model.multi_modal_projector
    projector.to(dtype=torch.bfloat16)

    # D.1 Remove Pooling
    if hasattr(projector, "avg_pool"):
        projector.avg_pool = torch.nn.Identity()

    # E. Load Trained Weights
    print(f"   Loading Adapter: {TRAINED_MODEL_PATH}")
    model = PeftModel.from_pretrained(model, TRAINED_MODEL_PATH)

    # D.2 FORCE PATCH SIZE UPDATE (The Missing Fix)
    # The projector still thinks it needs 64x64 patches. We force it to 27.
    print("   Forcing Projector Grid Size (64 -> 27)...")
    count = 0
    for m in model.modules():
        if hasattr(m, "patches_per_image"):
            if m.patches_per_image == 64:
                m.patches_per_image = 27
                count += 1
    print(f"   └── Updated {count} layers.")

    # Load Processor
    processor = AutoProcessor.from_pretrained(TRAINED_MODEL_PATH)
    processor.image_processor.size = {"height": TARGET_IMG_SIZE, "width": TARGET_IMG_SIZE}
    processor.image_processor.crop_size = {"height": TARGET_IMG_SIZE, "width": TARGET_IMG_SIZE}
    processor.image_processor.do_resize = True

    return model, processor

# =========================================================
# 3. GENERATION FUNCTION
# =========================================================
def generate_single_report(model, processor, image_path):
    try:
        # Handle Paths
        if "Seleccted_mimic_CXR_images" in image_path and not image_path.startswith("/"):
            image_path = os.path.join("/content", image_path)
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"[Error: {str(e)}]"

    prompt_text = "Describe the chest X-ray findings and impression."

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt_text}]}]
    text_input = processor.apply_chat_template(messages, add_generation_prompt=True)

    inputs = prepare_inputs(text_input, image, processor, device="cuda")

    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            pixel_values=inputs["pixel_values"],
            token_type_ids=inputs["token_type_ids"] if "token_type_ids" in inputs else None,
            max_new_tokens=200, do_sample=False, num_beams=1
        )

    return processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# =========================================================
# 4. MAIN WORKFLOW
# =========================================================
def main():
    # --- PHASE 1: GENERATION ---
    model, processor = load_evaluation_model()

    print(f"\n📂 Loading Test Data: {TEST_FILE_PATH}")
    with open(TEST_FILE_PATH, "r") as f:
        data = json.load(f)

    results = []
    print(f"🚀 Generating Reports for {len(data)} items...")

    for item in tqdm(data):
        img_path = item["messages"][0]["content"][0]["image"]
        ground_truth = item["messages"][1]["content"][0]["text"]

        prediction = generate_single_report(model, processor, img_path)

        results.append({
            "image_path": img_path,
            "ground_truth": ground_truth,
            "prediction": prediction
        })

    # SAVE TO CSV
    df = pd.DataFrame(results)
    df.to_csv(CSV_OUTPUT_PATH, index=False)
    print(f"\n💾 Saved {len(df)} predictions to: {CSV_OUTPUT_PATH}")
    print("   (You can download this file now to inspect results before metrics)")

if __name__ == "__main__":
    main()


In [ ]:
# 1. Downgrade transformers to make RadGraph work
#  RadGraph fails on transformers > 4.44 due to tokenizer changes
!pip uninstall -y transformers
!pip install -q transformers==4.40.0

# 2. Install evaluation libraries
!pip install -q evaluate rouge_score bert_score radgraph

import json
import numpy as np
import pandas as pd
import evaluate
from radgraph import F1RadGraph
from bert_score import score as bert_score_func
from tqdm import tqdm

# ---------------- CONFIG ----------------
PRED_FILE = "/content/NaijaCXR-VLM_predictions.csv"

# ---------------- LOAD DATA ----------------
print(f"Loading {PRED_FILE}...")
try:
        data = pd.read_csv(PRED_FILE)
        data = data.to_dict(orient='records')
except FileNotFoundError:
    print(f"❌ Error: {PRED_FILE} not found. Make sure you ran the inference script first.")
    data = []

if data:
    # Extract lists
    references = [item["ground_truth"] for item in data]
    predictions = [item["prediction"] for item in data]

    print(f"Evaluating {len(predictions)} reports...")

    # ---------------- 1. ROUGE & BLEU ----------------
    print("\nComputing ROUGE & BLEU...")
    rouge = evaluate.load("rouge")
    bleu = evaluate.load("bleu")

    rouge_res = rouge.compute(predictions=predictions, references=references)
    bleu_res = bleu.compute(predictions=predictions, references=references)

    print(f"✅ ROUGE-1: {rouge_res['rouge1']:.4f}")
    print(f"✅ ROUGE-L: {rouge_res['rougeL']:.4f}")
    print(f"✅ BLEU:    {bleu_res['bleu']:.4f}")

    # ---------------- 2. BERTScore ----------------
    print("\nComputing BERTScore (may download model)...")
    # BERTScore uses a pre-trained model to check semantic similarity
    P, R, F1 = bert_score_func(predictions, references, lang="en", verbose=False)
    bert_mean = F1.mean().item()
    print(f"✅ BERTScore F1: {bert_mean:.4f}")

    # ---------------- SUMMARY ----------------
    print("\n" + "="*30)
    print("🚀 FINAL RESULTS")
    print("="*30)
    print(f"ROUGE-L:      {rouge_res['rougeL']:.4f}")
    print(f"BLEU:         {bleu_res['bleu']:.4f}")
    print(f"BERTScore:    {bert_mean:.4f}")
    print("="*30)

import numpy as np
from radgraph import F1RadGraph
from tqdm import tqdm

# ---------------- 3. RadGraph F1 (Corrected) ----------------
print("\nComputing RadGraph F1 (Clinical Correctness)...")
try:
    # Initialize metric
    f1radgraph = F1RadGraph(reward_level="partial")

    # CRITICAL FIX: Capture all return values into one variable first
    # The function likely returns (f1_scores, precision, recall, something_else)
    radgraph_results = f1radgraph(hyps=predictions, refs=references)

    # The first item [0] is always the F1 score (either a list or a mean)
    f1_output = radgraph_results[0]

    # Ensure we get a single number (if it returns a list of scores, average them)
    radgraph_score = np.mean(f1_output)

    print(f"✅ RadGraph F1: {radgraph_score:.4f}")

except Exception as e:
    print(f"⚠️ RadGraph Failed: {e}")
    radgraph_score = 0.0

# ---------------- FINAL SUMMARY ----------------
print("\n" + "="*30)
print("🚀 FINAL RESULTS")
print("="*30)
print(f"RadGraph F1:  {radgraph_score:.4f}")
print("="*30)


## 11. Visual Grounding & Explainability (Grad-CAM)
Calculates transformer-level Grad-CAM activations from the SigLIP vision encoder for clinical findings (*cardiomegaly, opacity, effusion*).


In [ ]:
import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# =========================================================
# 1. GRAD-CAM FOR TRANSFORMER VISION ENCODER (SigLIP)
# =========================================================
class VLMGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        # Register hooks
        self.forward_hook = target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        if isinstance(output, tuple):
            self.activations = output[0].detach()
        else:
            self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        if isinstance(grad_output, tuple):
            self.gradients = grad_output[0].detach()
        else:
            self.gradients = grad_output.detach()

    def generate_heatmap(self, target_token_logit):
        # Zero out model gradients
        self.model.zero_grad()

        # Backward pass on target token logit
        target_token_logit.backward(retain_graph=True)

        if self.activations is None or self.gradients is None:
            raise RuntimeError("Activations or Gradients were not captured. Ensure gradient flow through the target layer.")

        # Extract activations and gradients
        act = self.activations
        grad = self.gradients
        if isinstance(act, tuple): act = act[0]
        if isinstance(grad, tuple): grad = grad[0]

        # Squeeze batch dimension if present
        if act.ndim == 3: act = act[0]
        if grad.ndim == 3: grad = grad[0]

        # Compute channel-wise weights (mean of gradients over spatial dimension)
        weights = grad.mean(dim=0) # [hidden_dim]

        # Weighted combination of activations
        heatmap = (act * weights).sum(dim=-1) # [seq_len]

        # Apply ReLU
        heatmap = F.relu(heatmap)

        # Normalize
        heatmap_min, heatmap_max = heatmap.min(), heatmap.max()
        if heatmap_max > heatmap_min:
            heatmap = (heatmap - heatmap_min) / (heatmap_max - heatmap_min)
        else:
            heatmap = torch.zeros_like(heatmap)

        # Reshape to 2D grid and interpolate to 384x384
        seq_len = heatmap.shape[0]
        grid_size = int(seq_len ** 0.5)

        # Handle cases where sequence length includes extra tokens (e.g. CLS)
        if grid_size * grid_size != seq_len:
            for offset in [1, 2]:
                new_seq_len = seq_len - offset
                new_grid = int(new_seq_len ** 0.5)
                if new_grid * new_grid == new_seq_len:
                    heatmap = heatmap[offset:]
                    grid_size = new_grid
                    break

        heatmap_2d = heatmap.view(1, 1, grid_size, grid_size)

        heatmap_resized = F.interpolate(
            heatmap_2d,
            size=(384, 384),
            mode='bilinear',
            align_corners=False
        ).squeeze()

        return heatmap_resized.cpu().numpy()

    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

# =========================================================
# 2. VISUALIZATION OVERLAY PLOTTER
# =========================================================
def plot_gradcam(image_path, heatmap, target_token_text, alpha=0.45):
    # Load and resize original image
    if isinstance(image_path, str):
        img = Image.open(image_path).convert("RGB")
    else:
        img = image_path

    img = np.array(img.resize((384, 384)))

    # Apply JET colormap to heatmap
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    # Overlay heatmap on original image
    overlayed = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)

    # Display results
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(img)
    axes[0].set_title("Original Chest X-Ray")
    axes[0].axis("off")

    axes[1].imshow(heatmap, cmap="jet")
    axes[1].set_title(f"Grad-CAM Heatmap: '{target_token_text}'")
    axes[1].axis("off")

    axes[2].imshow(overlayed)
    axes[2].set_title(f"Grounded Localization: '{target_token_text}'")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

# =========================================================
# 3. EXPLAINABLE REPORT GENERATOR
# =========================================================
def generate_grounded_report(model, processor, image_path, target_keywords=None):
    if target_keywords is None:
        target_keywords = ["effusion", "opacity", "cardiomegaly", "pneumothorax", "consolidation", "hilar", "infiltration"]

    # Generate report normally
    report = generate_report(model, processor, image_path)
    print("🩻 GENERATED REPORT:")
    print(report)
    print("-" * 50)

    # 1. Dynamically locate the SigLIP vision tower final encoder layer module
    target_layer = None

    # Recursive search for the SiglipVisionModel
    for module in model.modules():
        class_name = module.__class__.__name__
        if class_name == "SiglipVisionModel":
            if hasattr(module, "vision_model") and hasattr(module.vision_model, "encoder") and hasattr(module.vision_model.encoder, "layers"):
                if len(module.vision_model.encoder.layers) > 0:
                    target_layer = module.vision_model.encoder.layers[-1]
                    print(f"🎯 Found target layer in SiglipVisionModel: {module.vision_model.encoder.layers[-1]}")
                    break

    # Fallback to general modules containing 'encoder.layers' or named 'layers' under vision
    if target_layer is None:
        for name, module in model.named_modules():
            class_name = module.__class__.__name__
            if "SiglipEncoder" in class_name or "SiglipVisionTransformer" in class_name:
                if hasattr(module, "layers") and len(module.layers) > 0:
                    target_layer = module.layers[-1]
                    print(f"🎯 Found target layer in encoder/transformer: {module.layers[-1]}")
                    break

    # Fallback 2: checking nested attributes (including PEFT structure)
    if target_layer is None:
        current = model
        if hasattr(current, "base_model"):
            current = current.base_model
        if hasattr(current, "model"):
            current = current.model
        if hasattr(current, "vision_tower"):
            tower = current.vision_tower
            if hasattr(tower, "vision_model") and hasattr(tower.vision_model, "encoder") and hasattr(tower.vision_model.encoder, "layers"):
                target_layer = tower.vision_model.encoder.layers[-1]
                print(f"🎯 Found target layer via model attributes: {target_layer}")

    if target_layer is None:
        print("⚠️ Could not locate vision encoder layers for Grad-CAM.")
        return report

    gradcam = VLMGradCAM(model, target_layer)

    try:
        if isinstance(image_path, str):
            image = Image.open(image_path).convert("RGB")
        else:
            image = image_path

        # Match prompt with the fine-tuned/training prompt
        prompt_text = "Describe the chest X-ray findings and impression."
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt_text}]}]
        text_input = processor.apply_chat_template(messages, add_generation_prompt=True)
        full_text = text_input + report

        # Run forward pass with grads enabled
        with torch.enable_grad():
            inputs = prepare_inputs(full_text, image, processor, device="cuda")

            # Ensure pixel_values require gradients
            inputs["pixel_values"].requires_grad_(True)

            # Force target layer parameters to require grad to build the autograd graph
            for p in target_layer.parameters():
                p.requires_grad_(True)

            model.zero_grad()

            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pixel_values=inputs["pixel_values"],
                token_type_ids=inputs["token_type_ids"] if "token_type_ids" in inputs else None,
            )
            logits = outputs.logits # [batch, seq_len, vocab_size]

            input_ids_list = inputs["input_ids"][0].tolist()
            tokens_decoded = [processor.decode([tok_id], skip_special_tokens=True).strip().lower() for tok_id in input_ids_list]

            # Calculate prompt length using exact non-special report token decoding
            report_token_ids = processor(text=report, return_tensors="pt", add_special_tokens=False)["input_ids"][0].tolist()
            prompt_len = len(input_ids_list) - len(report_token_ids)

            found_any = False
            for keyword in target_keywords:
                matching_indices = [idx for idx, token in enumerate(tokens_decoded) if keyword in token]
                # Only consider generated tokens (after prompt length)
                matching_indices = [idx for idx in matching_indices if idx >= prompt_len]

                if matching_indices:
                    found_any = True
                    target_idx = matching_indices[0] # Ground first occurrence
                    target_token_id = input_ids_list[target_idx]

                    print(f"🔍 Grounding keyword '{keyword}' (Token '{tokens_decoded[target_idx]}' at position {target_idx})...")

                    target_logit = logits[0, target_idx, target_token_id]
                    heatmap = gradcam.generate_heatmap(target_logit)
                    plot_gradcam(image, heatmap, keyword)

            if not found_any:
                print("ℹ️ None of the target keywords were found in the generated report.")

    except Exception as e:
        print(f"⚠️ Grad-CAM failed: {e}")
    finally:
        gradcam.remove_hooks()

    return report


## 12. Zero-Shot Evaluation on VQA-RAD Chest Dataset


In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score

print("📂 Loading VQA-RAD dataset from Hugging Face...")
vqa_rad = load_dataset("flaviagiammarino/vqa-rad", split="test")
print(f"Loaded VQA-RAD test set with {len(vqa_rad)} examples.")

# Keywords to isolate chest cases
chest_keywords = [
    "chest", "lung", "heart", "pleural", "pneumothorax",
    "cardiomegaly", "effusion", "rib", "diaphragm", "infiltrate",
    "consolidation", "hilar", "mediastinum", "trachea"
]

def is_chest_question(example):
    q_text = str(example["question"]).lower()
    return any(kw in q_text for kw in chest_keywords)

chest_vqa = vqa_rad.filter(is_chest_question)
print(f"Filtered to {len(chest_vqa)} chest-related VQA examples.")

predictions_vqa = []
ground_truths_vqa = []
questions_vqa = []
closed_ended_accuracy = []

# Limit to first 100 questions for speed
limit_eval = min(100, len(chest_vqa))
print(f"Running VQA evaluation on first {limit_eval} chest VQA examples...")

model.eval()

for i in tqdm(range(limit_eval)):
    item = chest_vqa[i]
    question = item["question"]
    ground_truth = str(item["answer"]).strip()
    img = item["image"]

    # Query prompt format
    prompt_text = f"Question: {question} Answer concisely in one word or a short phrase."
    response = generate_report(model, processor, img, prompt=prompt_text)
    response_clean = response.strip()

    predictions_vqa.append(response_clean)
    ground_truths_vqa.append(ground_truth)
    questions_vqa.append(question)

    # Closed-ended (Yes/No) accuracy calculation
    gt_lower = ground_truth.lower()
    if gt_lower in ["yes", "no"]:
        resp_lower = response_clean.lower()
        resp_mapped = "yes" if "yes" in resp_lower else ("no" if "no" in resp_lower else "other")
        closed_ended_accuracy.append(1 if resp_mapped == gt_lower else 0)

if closed_ended_accuracy:
    closed_acc = np.mean(closed_ended_accuracy) * 100
    print(f"\n📊 Closed-Ended (Yes/No) Accuracy: {closed_acc:.2f}% ({len(closed_ended_accuracy)} questions)")
else:
    closed_acc = 0.0
    print("\n📊 No closed-ended Yes/No questions in the evaluated set.")

# Save VQA predictions
df_vqa = pd.DataFrame({
    "Question": questions_vqa,
    "Ground Truth": ground_truths_vqa,
    "Prediction": predictions_vqa
})
df_vqa.to_csv("vqa_rad_predictions.csv", index=False)
print("💾 Saved VQA-RAD predictions to 'vqa_rad_predictions.csv'")

print("\nComputing text generation metrics on open questions...")
from bert_score import score as bert_score_func
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

rouge_res = rouge.compute(predictions=predictions_vqa, references=ground_truths_vqa)
bleu_res = bleu.compute(predictions=predictions_vqa, references=ground_truths_vqa)
P, R, F1 = bert_score_func(predictions_vqa, ground_truths_vqa, lang="en", verbose=False)
bert_mean = F1.mean().item()

print("\n" + "="*30)
print("🚀 VQA-RAD CHEST RESULTS")
print("="*30)
print(f"Closed-Ended Yes/No Acc:  {closed_acc:.2f}%")
print(f"ROUGE-L:                 {rouge_res['rougeL']:.4f}")
print(f"BLEU:                    {bleu_res['bleu']:.4f}")
print(f"BERTScore F1:            {bert_mean:.4f}")
print("="*30)
